# Profile-coherence CrossEncoder evaluation

This notebook evaluates only the demographic profile-coherence aligner. It compares the trained CrossEncoder with held-out Qwen-generated scores and reports ranking, error, discrimination, and calibration metrics.

It does not evaluate POI, activity, schedule, or vehicle alignment.

In [1]:
from pathlib import Path
import importlib.util
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from scipy.stats import kendalltau, spearmanr
    from sklearn.metrics import average_precision_score, mean_absolute_error, mean_squared_error, roc_auc_score, roc_curve
except ImportError as exc:
    raise RuntimeError('Install the notebook dependencies, including scipy and scikit-learn.') from exc

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'pyproject.toml').exists():
    repo_root = repo_root.parent
if not (repo_root / 'pyproject.toml').exists():
    raise FileNotFoundError('Run this notebook from inside the citybehavex repository.')

output_dir = repo_root / 'notebooks/06_profile_coherence_evaluation/outputs'
output_dir.mkdir(parents=True, exist_ok=True)

# Set this explicitly if you want a particular scenario. Otherwise the notebook
# selects the first profile artifact that already exists.
PROFILES_PATH = None
PROFILE_CANDIDATES = [
    repo_root / 'data/shanghai/shanghai_agent_profiles.parquet',
    repo_root / 'data/yjmob/yjmob_agent_profiles.parquet',
    repo_root / 'data/yjmob2/yjmob2_agent_profiles.parquet',
    repo_root / 'data/gparis/results/gparis_agent_profiles.parquet',
    repo_root / 'data/la/results/la_1pct_agent_profiles.parquet',
]
MODEL_PATH = repo_root / 'models/modernbert-profile-coherence-aligner'
EVAL_LABELS_PATH = output_dir / 'profile_coherence_eval_scores.parquet'
TRAINING_LABELS_PATH = repo_root / 'data/profile_coherence_scores.parquet'
LLM_BASE_URL = 'http://localhost:8081/v1'
LLM_MODEL = 'Qwen/Qwen2.5-32B-Instruct-AWQ'
SAMPLE_SIZE = 2000
MUTATION_RATIO = 0.5
SEED = 42
TEST_FRACTION = 0.2
LLM_CONCURRENCY = 8
DEVICE = 'cuda'


## Load the project implementation

The notebook imports the existing training script rather than duplicating its profile narrative, mutation, prompt, and labeling logic.

In [2]:
if PROFILES_PATH is None:
    existing_profiles = [path for path in PROFILE_CANDIDATES if path.exists()]
    if not existing_profiles:
        configured_paths = '\n'.join(f'  - {path}' for path in PROFILE_CANDIDATES)
        raise FileNotFoundError(
            'No generated profile artifact was found. Run a scenario first, '
            'or set PROFILES_PATH to an existing profile Parquet file. Checked:\n'
            + configured_paths
        )
    PROFILES_PATH = existing_profiles[0]
print(f'Using profile input: {PROFILES_PATH}')
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f'Profile-coherence checkpoint not found: {MODEL_PATH}. '
        'Train scripts/train_modernbert_profile_coherence_aligner.py first.'
    )

script_path = repo_root / 'scripts/train_modernbert_profile_coherence_aligner.py'
spec = importlib.util.spec_from_file_location('profile_coherence_training', script_path)
training = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(training)

profiles = training.load_profiles(PROFILES_PATH)
profile_ids = np.array(sorted({int(profile.uid) for profile in profiles}), dtype=np.int64)
print(f'Loaded {len(profiles):,} profiles ({len(profile_ids):,} unique profile IDs).')


Using profile input: /home/gustavo/citybehavex/data/shanghai/shanghai_agent_profiles.parquet


FileNotFoundError: Profile-coherence checkpoint not found: /home/gustavo/citybehavex/models/modernbert-profile-coherence-aligner. Train scripts/train_modernbert_profile_coherence_aligner.py first.

## Build an evaluation set

If cached evaluation labels exist, they are reused. Otherwise, profile IDs present in the training label dataset are excluded when possible, and fresh profile variants are labeled by Qwen. This keeps the evaluation at profile level rather than randomly splitting repeated rows.

In [ ]:
def load_labels(path):
    if not path.exists():
        return None
    return pd.read_parquet(path) if path.suffix == '.parquet' else pd.read_json(path, orient='records', lines=True)

eval_df = load_labels(EVAL_LABELS_PATH)
if eval_df is None:
    training_df = load_labels(TRAINING_LABELS_PATH)
    training_ids = set() if training_df is None else set(training_df['profile_uid'].astype(int))
    held_out_ids = np.array([uid for uid in profile_ids if uid not in training_ids], dtype=np.int64)
    if len(held_out_ids) < 2:
        rng = np.random.default_rng(SEED)
        shuffled = profile_ids.copy()
        rng.shuffle(shuffled)
        n_test = max(2, int(round(len(shuffled) * TEST_FRACTION)))
        held_out_ids = shuffled[:n_test]
        warnings.warn('Could not identify unused profile IDs; using a deterministic profile-level holdout instead.')
    held_out = [profile for profile in profiles if int(profile.uid) in set(held_out_ids)]
    pairs = training.build_training_pairs(
        held_out, sample_size=SAMPLE_SIZE, mutation_ratio=MUTATION_RATIO, seed=SEED
    )
    eval_df = training.label_pairs(
        pairs, base_url=LLM_BASE_URL, model=LLM_MODEL, api_key=None,
        timeout=120.0, retries=3, concurrency=LLM_CONCURRENCY, progress_interval=100
    )
    eval_df.to_parquet(EVAL_LABELS_PATH, index=False)
    print(f'Generated and cached {len(eval_df):,} held-out labels at {EVAL_LABELS_PATH}.')
else:
    print(f'Loaded {len(eval_df):,} cached evaluation labels from {EVAL_LABELS_PATH}.')

required_columns = {'profile_uid', 'variant', 'context_text', 'candidate_text', 'score'}
missing = required_columns - set(eval_df.columns)
if missing:
    raise ValueError(f'Evaluation labels are missing columns: {sorted(missing)}')
eval_df = eval_df.dropna(subset=['context_text', 'candidate_text', 'score']).copy()
eval_df['score'] = eval_df['score'].astype(float).clip(0.0, 1.0)
display(eval_df[['profile_uid', 'variant', 'score']].head())
display(eval_df['variant'].value_counts().rename('count').to_frame())


## Predict with the trained CrossEncoder

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder

model = CrossEncoder(str(MODEL_PATH), device=DEVICE)
pairs = eval_df[['context_text', 'candidate_text']].astype(str).values.tolist()
eval_df['prediction'] = np.asarray(model.predict(pairs, batch_size=128, show_progress_bar=True), dtype=float).reshape(-1)
eval_df['prediction'] = eval_df['prediction'].clip(0.0, 1.0)
display(eval_df[['variant', 'score', 'prediction']].head())


## Metrics

Correlation and ranking metrics measure agreement with the teacher. MAE/RMSE measure score error. ROC-AUC and PR-AUC test whether the model separates original profiles from deliberately mutated profiles. Pairwise accuracy compares original and mutated examples from the same profile ID.

In [ ]:
y_true = eval_df['score'].to_numpy()
y_pred = eval_df['prediction'].to_numpy()
metrics = {
    'n_examples': int(len(eval_df)),
    'n_profiles': int(eval_df['profile_uid'].nunique()),
    'spearman': float(spearmanr(y_true, y_pred).statistic),
    'kendall_tau': float(kendalltau(y_true, y_pred).statistic),
    'mae': float(mean_absolute_error(y_true, y_pred)),
    'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
}

binary = eval_df['variant'].eq('original').astype(int).to_numpy()
if len(np.unique(binary)) == 2:
    metrics['original_vs_mutated_roc_auc'] = float(roc_auc_score(binary, eval_df['prediction']))
    metrics['original_vs_mutated_pr_auc'] = float(average_precision_score(binary, eval_df['prediction']))
else:
    metrics['original_vs_mutated_roc_auc'] = None
    metrics['original_vs_mutated_pr_auc'] = None

original = eval_df[eval_df['variant'].eq('original')].groupby('profile_uid')['prediction'].mean()
mutated = eval_df[eval_df['variant'].eq('mutated')].groupby('profile_uid')['prediction'].mean()
common = original.index.intersection(mutated.index)
metrics['paired_original_beats_mutated'] = (
    float((original.loc[common] > mutated.loc[common]).mean()) if len(common) else None
)

calibration_bins = pd.DataFrame({'prediction': y_pred, 'teacher': y_true})
calibration_bins['bin'] = pd.cut(calibration_bins['prediction'], bins=np.linspace(0, 1, 11), include_lowest=True)
calibration = calibration_bins.groupby('bin', observed=False).agg(prediction=('prediction', 'mean'), teacher=('teacher', 'mean'), count=('teacher', 'size')).dropna()
metrics['calibration_mae'] = float(np.average(np.abs(calibration['prediction'] - calibration['teacher']), weights=calibration['count'])) if len(calibration) else None

metrics_df = pd.DataFrame([metrics]).T.rename(columns={0: 'value'})
display(metrics_df)
with (output_dir / 'profile_coherence_metrics.json').open('w', encoding='utf-8') as handle:
    json.dump(metrics, handle, indent=2)
eval_df.to_parquet(output_dir / 'profile_coherence_predictions.parquet', index=False)


## Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].scatter(y_true, y_pred, alpha=0.35, s=14)
axes[0].plot([0, 1], [0, 1], '--', color='black')
axes[0].set(xlabel='Qwen score', ylabel='CrossEncoder score', title='Teacher vs model')

for variant, group in eval_df.groupby('variant'):
    axes[1].hist(group['prediction'], bins=20, alpha=0.55, label=variant)
axes[1].set(xlabel='CrossEncoder score', ylabel='count', title='Prediction distributions')
axes[1].legend()

axes[2].scatter(y_pred, y_pred - y_true, alpha=0.35, s=14)
axes[2].axhline(0, linestyle='--', color='black')
axes[2].set(xlabel='CrossEncoder score', ylabel='prediction - teacher', title='Residuals')
plt.tight_layout()
plt.savefig(output_dir / 'profile_coherence_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

plt.figure(figsize=(5, 4))
plt.plot(calibration['prediction'], calibration['teacher'], 'o-', label='observed')
plt.plot([0, 1], [0, 1], '--', color='black', label='ideal')
plt.xlabel('Mean predicted score')
plt.ylabel('Mean Qwen score')
plt.title('Score calibration')
plt.legend()
plt.tight_layout()
plt.savefig(output_dir / 'profile_coherence_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

if len(np.unique(binary)) == 2:
    fpr, tpr, _ = roc_curve(binary, eval_df['prediction'])
    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, label=f"ROC-AUC={metrics['original_vs_mutated_roc_auc']:.3f}")
    plt.plot([0, 1], [0, 1], '--', color='black')
    plt.xlabel('False-positive rate')
    plt.ylabel('True-positive rate')
    plt.title('Original vs mutated profiles')
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_dir / 'profile_coherence_roc.png', dpi=150, bbox_inches='tight')
    plt.show()
